# Azure Content Understanding with LangChain

This notebook demonstrates how to use `AzureAIContentUnderstandingLoader` from the `langchain-azure-ai` package to load and analyze documents, images, audio, and video through [Azure Content Understanding](https://learn.microsoft.com/en-us/azure/ai-services/content-understanding/) (CU).

CU is a multimodal AI service that produces structured markdown and extracted fields from unstructured content -- with confidence scores, layout preservation, and support for all four modalities under a single API.

## Prerequisites

1. An Azure AI Foundry resource with a Content Understanding endpoint
2. Model deployments: `gpt-4.1-mini`/`gpt-4.1` and `text-embedding-3-large` (required for prebuilt search analyzers and custom analyzer examples in sections 1 and 4b)
3. [Configure default model deployments](https://learn.microsoft.com/en-us/azure/ai-services/content-understanding/concepts/models-deployments?tabs=rest-api) on your resource
4. Authentication credentials (Azure Entra ID or API key)


In [ ]:
# Install the package:
%pip install -U langchain-azure-ai

In [ ]:
import os

from langchain_azure_ai.document_loaders import AzureAIContentUnderstandingLoader

# Set your CU endpoint
os.environ["CONTENTUNDERSTANDING_ENDPOINT"] = "https://<your-resource>.services.ai.azure.com/"

# If using API key authentication (skip if using DefaultAzureCredential):
# os.environ["CONTENTUNDERSTANDING_KEY"] = "<your-api-key>"

In [ ]:
from azure.identity import DefaultAzureCredential

endpoint = os.environ["CONTENTUNDERSTANDING_ENDPOINT"]
credential = DefaultAzureCredential()

# Or use an API key:
# credential = os.environ["CONTENTUNDERSTANDING_KEY"]

In [ ]:
# Sample assets (public GitHub-hosted files for demo purposes)
ASSET_BASE = "https://raw.githubusercontent.com/Azure-Samples/azure-ai-content-understanding-assets/main"
SAMPLE_PDF_URL = f"{ASSET_BASE}/document/invoice.pdf"
SAMPLE_IMAGE_URL = f"{ASSET_BASE}/image/pieChart.jpg"
SAMPLE_AUDIO_URL = f"{ASSET_BASE}/audio/callCenterRecording.mp3"
SAMPLE_VIDEO_URL = f"{ASSET_BASE}/videos/sdk_samples/FlightSimulator.mp4"
SAMPLE_MULTI_PAGE_PDF_URL = f"{ASSET_BASE}/document/mixed_financial_docs.pdf"

## 1. PDF Document — Rich Markdown / Field Extraction / Custom Analyzer

The most common use case. CU returns clean markdown with preserved layout (tables, headings) and structured field extraction with confidence scores.

Unlike `PyPDFLoader` (plain text) or `PyMuPDFLoader` (appended tables), CU produces **inline markdown tables** and **structured fields with confidence**. This section also demonstrates custom field extraction with `prebuilt-invoice` and a custom analyzer using `field_schema`.

In [ ]:
# Option 1: Load from a URL (public or SAS-signed)
# When no analyzer_id is specified, the loader auto-selects a prebuilt-*Search analyzer
# based on the file's modality (e.g., prebuilt-documentSearch for PDFs and image files).
loader = AzureAIContentUnderstandingLoader(
    endpoint=endpoint,
    credential=credential,
    url=SAMPLE_PDF_URL,
)

# Option 2: Load from a local file path
# loader = AzureAIContentUnderstandingLoader(
#     endpoint=endpoint,
#     credential=credential,
#     file_path="path/to/local/invoice.pdf",
# )

# Option 3: Load from raw bytes (e.g., database blob, HTTP response, in-memory buffer)
# loader = AzureAIContentUnderstandingLoader(
#     endpoint=endpoint,
#     credential=credential,
#     bytes_source=pdf_bytes,
#     source="my-invoice.pdf",  # custom label for metadata["source"]
# )

# Exactly one of url, file_path, or bytes_source must be provided.
# For async usage, call `await loader.aload()` instead of `loader.load()`.
docs = loader.load()

print(f"Loaded {len(docs)} document(s)")
print(f"Kind: {docs[0].metadata['kind']}")
print(f"Analyzer: {docs[0].metadata['analyzer_id']}")
print()
print("--- Markdown preview (first 500 chars) ---")
print(docs[0].page_content[:500])

In [ ]:
# Examine extracted fields with confidence scores.
# With the default prebuilt-*Search analyzer, only a "Summary" field is returned.
if "fields" in docs[0].metadata:
    fields = docs[0].metadata["fields"]
    print(f"Extracted {len(fields)} field(s):")
    for name, field_data in fields.items():
        if isinstance(field_data, dict):
            print(f"  {name}: {field_data.get('value')} "
                  f"(type={field_data.get('type')}, confidence={field_data.get('confidence')})")
        elif isinstance(field_data, list):
            print(f"  {name}: [{len(field_data)} items]")
else:
    print("No fields extracted (try a custom analyzer with field schema)")

### Field Extraction with Prebuilt Analyzers

Use a domain-specific prebuilt analyzer (like `prebuilt-invoice`) to extract structured fields with confidence scores. Domain-specific analyzers require `model_deployments` to specify which model deployment to use. See the full list of [prebuilt analyzers](https://learn.microsoft.com/azure/ai-services/content-understanding/concepts/prebuilt-analyzers).

You can also create your own custom analyzer with a user-defined field schema for any modality — see the [custom analyzer guide](https://learn.microsoft.com/en-us/azure/ai-services/content-understanding/tutorial/create-custom-analyzer?tabs=portal%2Cdocument).

In [ ]:
loader = AzureAIContentUnderstandingLoader(
    endpoint=endpoint,
    credential=credential,
    analyzer_id="prebuilt-invoice",  # or replace with your custom analyzer ID
    url=SAMPLE_PDF_URL,
    model_deployments={"gpt-4.1": "gpt-4.1"},  # map model name -> your deployment name
)
docs = loader.load()

fields = docs[0].metadata.get("fields", {})
for name, data in fields.items():
    if isinstance(data, dict):
        print(f"{name}: {data['value']} (confidence: {data.get('confidence', 'N/A')})")

### Custom Document Analyzer with Field Schema

Use a **custom analyzer** with `field_schema` to extract structured fields, generate summaries, and classify documents. The extracted fields appear in `doc.metadata["fields"]`.

This requires a custom analyzer created with `field_schema` and `base_analyzer_id="prebuilt-document"`. Example config:

```python
# Create the analyzer via the CU SDK (one-time setup):
from azure.ai.contentunderstanding.models import (
    ContentAnalyzer, ContentAnalyzerConfig,
    ContentFieldSchema, ContentFieldDefinition,
    ContentFieldType, GenerationMethod,
)

analyzer = ContentAnalyzer(
    base_analyzer_id="prebuilt-document",
    config=ContentAnalyzerConfig(
        enable_ocr=True,
        estimate_field_source_and_confidence=True,
        return_details=True,
    ),
    field_schema=ContentFieldSchema(
        name="document_schema",
        description="Schema for extracting document information",
        fields={
            "company_name": ContentFieldDefinition(
                type=ContentFieldType.STRING,
                method=GenerationMethod.EXTRACT,
                description="Name of the company",
            ),
            "total_amount": ContentFieldDefinition(
                type=ContentFieldType.NUMBER,
                method=GenerationMethod.EXTRACT,
                description="Total amount on the document",
            ),
            "document_summary": ContentFieldDefinition(
                type=ContentFieldType.STRING,
                method=GenerationMethod.GENERATE,
                description="A brief summary of the document content",
            ),
            "document_type": ContentFieldDefinition(
                type=ContentFieldType.STRING,
                method=GenerationMethod.CLASSIFY,
                description="Type of document",
                enum=["invoice", "bank_statement", "loan_application", "other"],
            ),
        },
    ),
    models={"completion": "gpt-4.1", "embedding": "text-embedding-3-large"},
)
client.begin_create_analyzer(analyzer_id="my_invoice_analyzer", resource=analyzer).result()
```

In [ ]:
# Custom document analyzer with field_schema (see config above)
loader = AzureAIContentUnderstandingLoader(
    endpoint=endpoint,
    credential=credential,
    analyzer_id="my_invoice_analyzer",  # replace with your custom analyzer ID
    url=SAMPLE_PDF_URL,
)
docs = loader.load()

print(f"Loaded {len(docs)} document(s)")
fields = docs[0].metadata.get("fields", {})
for name, value in fields.items():
    print(f"  {name}: {value}")

### Document Classification with Segment Mode

Use `output_mode="segment"` with a custom classifier analyzer to split a multi-page document by category. Each classified segment becomes a separate LangChain `Document` with its own `category`, page range, and markdown.

This requires a custom analyzer with `base_analyzer_id="prebuilt-document"`, `content_categories`, and `enable_segment=True`. Example config:

```python
# Create the analyzer via the CU SDK (one-time setup):
from azure.ai.contentunderstanding.models import (
    ContentAnalyzer, ContentAnalyzerConfig, ContentCategoryDefinition,
)

analyzer = ContentAnalyzer(
    base_analyzer_id="prebuilt-document",
    config=ContentAnalyzerConfig(
        enable_segment=True,
        return_details=True,
        enable_ocr=True,
        content_categories={
            "Invoice": ContentCategoryDefinition(
                description="Invoices, bills, and payment requests",
                analyzer_id="prebuilt-document",
            ),
            "BankStatement": ContentCategoryDefinition(
                description="Bank account statements and transaction records",
                analyzer_id="prebuilt-document",
            ),
            "LoanApplication": ContentCategoryDefinition(
                description="Loan applications and credit requests",
                analyzer_id="prebuilt-document",
            ),
        },
    ),
    models={"completion": "gpt-4.1"},
)
client.begin_create_analyzer(analyzer_id="my_doc_classifier", resource=analyzer).result()
```

> **Note:** Document classification uses `prebuilt-document` as the base. For video segmentation, use `prebuilt-video` as the base — see section 4b above.

In [ ]:
# Document classification + segmentation (requires a custom classifier — see config above)
loader = AzureAIContentUnderstandingLoader(
    endpoint=endpoint,
    credential=credential,
    analyzer_id="my_doc_classifier",  # replace with your custom classifier ID
    url=SAMPLE_MULTI_PAGE_PDF_URL,
    output_mode="segment",
)
docs = loader.load()

print(f"Found {len(docs)} segment(s):")
for doc in docs:
    cat = doc.metadata.get('category', 'N/A')
    start_pg = doc.metadata.get('start_page_number', '?')
    end_pg = doc.metadata.get('end_page_number', '?')
    print(f"  [{cat}] pages {start_pg}-{end_pg}: {doc.page_content[:80]}...")

## 2. Image -- Chart / Scanned Document

By default, the loader uses `prebuilt-documentSearch` for images — treating them as documents with OCR and layout analysis. This works well for charts, scanned documents, and handwritten text.

To use image-specific analysis instead, override with `analyzer_id="prebuilt-imageSearch"`.

In [ ]:
# Default: uses prebuilt-documentSearch (treats image as document)
loader = AzureAIContentUnderstandingLoader(
    endpoint=endpoint,
    credential=credential,
    url=SAMPLE_IMAGE_URL,
    # analyzer_id="prebuilt-imageSearch",  # uncomment to use image-specific analyzer instead
)
docs = loader.load()

print(f"Kind: {docs[0].metadata['kind']}")    # "document" (images return document kind)
print(f"Source: {docs[0].metadata['source']}")
print()
print("--- Extracted text ---")
print(docs[0].page_content)

## 3. Audio Transcription

CU processes audio files and returns timestamped transcripts as markdown. This enables RAG pipelines over call recordings, podcasts, and meetings.

> **Note:** Audio content is always returned as a single transcript. Segmentation (`enableSegment`) is **not** supported for audio-based analyzers (`prebuilt-audio`, `prebuilt-callCenter`) — the service rejects the configuration. Use `output_mode="markdown"` (default) for audio.

In [ ]:
loader = AzureAIContentUnderstandingLoader(
    endpoint=endpoint,
    credential=credential,
    url=SAMPLE_AUDIO_URL,
)
docs = loader.load()

print(f"Kind: {docs[0].metadata['kind']}")              # "audioVisual"
print(f"Duration: {docs[0].metadata['start_time_ms']}ms -- {docs[0].metadata['end_time_ms']}ms")
print()
print("--- Transcript preview (first 500 chars) ---")
print(docs[0].page_content[:500])

## 4. Video — Transcript and Segmentation

Videos are auto-detected and analyzed with `prebuilt-videoSearch`, returning a transcript as markdown.

For **video segmentation** — splitting a video into topic-based segments — see section 4b below. This requires a custom analyzer with `enableSegment=true`.

In [ ]:
# Basic video transcript (auto-selects prebuilt-videoSearch)
loader = AzureAIContentUnderstandingLoader(
    endpoint=endpoint,
    credential=credential,
    url=SAMPLE_VIDEO_URL,
)
docs = loader.load()

print(f"Kind: {docs[0].metadata['kind']}")
print(f"Duration: {docs[0].metadata.get('start_time_ms', '?')}ms -- {docs[0].metadata.get('end_time_ms', '?')}ms")
print()
print("--- Transcript preview (first 500 chars) ---")
print(docs[0].page_content[:500])

### 4b. Video Segmentation with Segment Mode

Use `output_mode="segment"` with a custom video segmenter analyzer to split a video into topic-based segments. Each segment becomes a separate LangChain `Document` with its own transcript, time range, category, and optional custom fields.

This requires a custom analyzer with `base_analyzer_id="prebuilt-video"`, `enable_segment=True`, and `content_categories` referencing a sub-analyzer. Example config:

```python
# Create the analyzer via the CU SDK (one-time setup):
from azure.ai.contentunderstanding.models import (
    ContentAnalyzer, ContentAnalyzerConfig, ContentCategoryDefinition,
)

# Option 1: Use prebuilt-videoSynopsis for a Summary field per segment
analyzer = ContentAnalyzer(
    base_analyzer_id="prebuilt-video",
    config=ContentAnalyzerConfig(
        enable_segment=True,
        content_categories={
            "Segment": ContentCategoryDefinition(
                description="A topic-based segment of the video",
                analyzer_id="prebuilt-videoSynopsis",
            ),
        },
    ),
    models={"completion": "gpt-4.1"},
)
client.begin_create_analyzer(analyzer_id="my_video_segmenter", resource=analyzer).result()

# Option 2: Use a custom sub-analyzer for richer per-segment fields
# First create the sub-analyzer:
sub_analyzer = ContentAnalyzer(
    base_analyzer_id="prebuilt-video",
    field_schema=ContentFieldSchema(
        name="segment_fields",
        description="Per-segment extraction",
        fields={
            "Title": ContentFieldDefinition(type=ContentFieldType.STRING, method=GenerationMethod.GENERATE,
                description="A concise title for this segment"),
            "Summary": ContentFieldDefinition(type=ContentFieldType.STRING, method=GenerationMethod.GENERATE,
                description="A brief summary of the segment content"),
            "Topics": ContentFieldDefinition(type=ContentFieldType.STRING, method=GenerationMethod.GENERATE,
                description="Comma-separated list of topics discussed"),
        },
    ),
    models={"completion": "gpt-4.1"},
)
client.begin_create_analyzer(analyzer_id="my_video_sub_analyzer", resource=sub_analyzer).result()

# Then reference it in the segmenter:
segmenter = ContentAnalyzer(
    base_analyzer_id="prebuilt-video",
    config=ContentAnalyzerConfig(
        enable_segment=True,
        content_categories={
            "Segment": ContentCategoryDefinition(
                description="A topic-based segment of the video",
                analyzer_id="my_video_sub_analyzer",
            ),
        },
    ),
    models={"completion": "gpt-4.1"},
)
client.begin_create_analyzer(analyzer_id="my_video_segmenter_v2", resource=segmenter).result()
```

> **Segmentation support by modality:**
> - ✅ **Document** (`prebuilt-document`): Produces `DocumentContentSegment` — each segment has page ranges and category
> - ✅ **Video** (`prebuilt-video`): Produces `AudioVisualContentSegment` — each segment has time ranges and optional fields
> - ❌ **Audio** (`prebuilt-audio`, `prebuilt-callCenter`): **Not supported** — the service rejects `enableSegment=true`
>
> **Nesting depth limit:** Sub-analyzers must not themselves contain sub-analyzers. Max depth is 2 (parent → sub-analyzer). For example, `prebuilt-videoSearch` cannot be used as a sub-analyzer because it internally already references `prebuilt-videoSynopsis`.

In [ ]:
# Video segmentation (requires a custom video segmenter — see config above)
loader = AzureAIContentUnderstandingLoader(
    endpoint=endpoint,
    credential=credential,
    analyzer_id="my_video_segmenter",  # replace with your video segmenter ID
    url=SAMPLE_VIDEO_URL,
    output_mode="segment",
)
docs = loader.load()

print(f"Found {len(docs)} segment(s):")
for doc in docs:
    cat = doc.metadata.get("category", "N/A")
    start = doc.metadata.get("start_time_ms", "?")
    end = doc.metadata.get("end_time_ms", "?")
    fields = doc.metadata.get("fields", {})
    field_names = ", ".join(fields.keys()) if fields else "none"
    print(f"  [{cat}] {start}ms–{end}ms | fields: {field_names}")
    print(f"    {doc.page_content[:100]}...")
    print()

## 5. Page Mode -- Per-Page Documents for RAG

Use `output_mode="page"` to get one `Document` per page. Each page carries `metadata["page"]` for source attribution in RAG pipelines.

> **Note:** Page mode is for document content only (not audio/video) and intentionally omits document-level fields — each page contains only its markdown text. Use the default `"markdown"` mode if you need extracted fields.

In [ ]:
loader = AzureAIContentUnderstandingLoader(
    endpoint=endpoint,
    credential=credential,
    url=SAMPLE_MULTI_PAGE_PDF_URL,
    output_mode="page",
)
docs = loader.load()

print(f"Loaded {len(docs)} page(s)")
for doc in docs[:3]:
    print(f"\n--- Page {doc.metadata['page']} ---")
    print(doc.page_content[:200])

## 6. Content Range -- Analyze a Subset

Use `content_range` to analyze specific pages (documents) or time ranges (audio/video) without processing the entire file.

In [ ]:
# Analyze only pages 1-3 of a multi-page PDF
loader = AzureAIContentUnderstandingLoader(
    endpoint=endpoint,
    credential=credential,
    url=SAMPLE_MULTI_PAGE_PDF_URL,
    content_range="1-3",
)
docs = loader.load()
print(f"Analyzed {len(docs)} document(s) from selected pages")
print(docs[0].page_content[:300])

In [ ]:
# Analyze the first 60 seconds of an audio file
loader = AzureAIContentUnderstandingLoader(
    endpoint=endpoint,
    credential=credential,
    url=SAMPLE_AUDIO_URL,
    content_range="0-60000",  # milliseconds
)
docs = loader.load()
print(f"Transcript (first 60s): {docs[0].page_content[:300]}")

## 7. Async Concurrent Loading

The loader implements true async via CU's native async client. Use `aload()` with `asyncio.gather()` to analyze multiple files concurrently -- zero blocked threads.

In [ ]:
import asyncio

async def load_concurrently():
    loaders = [
        AzureAIContentUnderstandingLoader(
            endpoint=endpoint, credential=credential,
            url=SAMPLE_PDF_URL,
        ),
        AzureAIContentUnderstandingLoader(
            endpoint=endpoint, credential=credential,
            url=SAMPLE_AUDIO_URL,
        ),
    ]
    # True async -- concurrent CU analysis, no blocked threads
    results = await asyncio.gather(*[loader.aload() for loader in loaders])
    for docs in results:
        print(f"{len(docs)} doc(s), kind={docs[0].metadata['kind']}")
    return results

results = await load_concurrently()

## 8. Process Multiple Files and Ask a Question

Load several files (even mixed modalities), combine their content, and ask an LLM for a summary.

In [ ]:
from azure.identity import get_bearer_token_provider
from langchain_openai import AzureChatOpenAI

# Load multiple files
all_docs = []
for url in [SAMPLE_PDF_URL, SAMPLE_MULTI_PAGE_PDF_URL]:
    loader = AzureAIContentUnderstandingLoader(
        endpoint=endpoint,
        credential=credential,
        url=url,
    )
    all_docs.extend(loader.load())

print(f"Loaded {len(all_docs)} document(s) from {len(set(d.metadata['source'] for d in all_docs))} file(s)")

# Combine and ask an LLM
combined = "\n\n---\n\n".join(
    f"[{doc.metadata['source']}]\n{doc.page_content}" for doc in all_docs
)

token_provider = get_bearer_token_provider(
    DefaultAzureCredential(),
    "https://cognitiveservices.azure.com/.default",
)

llm = AzureChatOpenAI(
    azure_deployment="gpt-4o",
    azure_endpoint="https://<your-openai-resource>.openai.azure.com",
    azure_ad_token_provider=token_provider,
    api_version="2025-01-01-preview",
)

answer = llm.invoke(f"Compare these documents and summarize key differences:\n\n{combined}")
print(answer.content)

## 9. Full RAG Pipeline

For large documents (hundreds of pages) or long audio/video, use LangChain's RAG ecosystem:
1. Load via CU with page-level splitting
2. Split pages into smaller chunks
3. Embed and index in a vector store
4. Build a retrieval-augmented QA chain

In [ ]:
# Additional packages for RAG pipeline (sections 8-9)
%pip install -U langchain-openai langchain-text-splitters langchain-community faiss-cpu

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import AzureOpenAIEmbeddings, AzureChatOpenAI
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

# 1. Load with page-level splitting
loader = AzureAIContentUnderstandingLoader(
    endpoint=endpoint,
    credential=credential,
    url=SAMPLE_MULTI_PAGE_PDF_URL,
    output_mode="page",
)
docs = loader.load()
print(f"Loaded {len(docs)} page(s)")

# 2. Split into smaller chunks
splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
chunks = splitter.split_documents(docs)
print(f"Split into {len(chunks)} chunk(s)")

# 3. Embed and index
token_provider = get_bearer_token_provider(
    DefaultAzureCredential(),
    "https://cognitiveservices.azure.com/.default",
)

embeddings = AzureOpenAIEmbeddings(
    azure_deployment="text-embedding-ada-002",
    azure_endpoint="https://<your-openai-resource>.openai.azure.com",
    azure_ad_token_provider=token_provider,
)
vectorstore = FAISS.from_documents(chunks, embeddings)

# 4. Retrieval-augmented QA
llm = AzureChatOpenAI(
    azure_deployment="gpt-4o",
    azure_endpoint="https://<your-openai-resource>.openai.azure.com",
    azure_ad_token_provider=token_provider,
    api_version="2025-01-01-preview",
)
retriever = vectorstore.as_retriever()
prompt = ChatPromptTemplate.from_template(
    "Answer the question based on the context below.\n\nContext:\n{context}\n\nQuestion: {question}"
)

rag_chain = (
    {"context": retriever | (lambda docs: "\n\n".join(d.page_content for d in docs)),
     "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

answer = rag_chain.invoke("What were the key risk factors mentioned in the report?")
print(answer)

## 10. Bytes Input with Custom Source Label

Load content from raw bytes (e.g., from a web download or database) with a custom `source` label for metadata tracking.

In [ ]:
# Download a file as bytes and load via bytes_source
import urllib.request
pdf_bytes = urllib.request.urlopen(SAMPLE_PDF_URL).read()

loader = AzureAIContentUnderstandingLoader(
    endpoint=endpoint,
    credential=credential,
    bytes_source=pdf_bytes,
    source="uploaded-invoice.pdf",  # custom label for metadata["source"]
)
docs = loader.load()

print(f"Source: {docs[0].metadata['source']}")  # "uploaded-invoice.pdf"
print(f"Content: {docs[0].page_content[:200]}")

## 11. Using analyze_kwargs for Advanced Options

Pass additional keyword arguments to the CU SDK's `begin_analyze` method -- for example, `processing_location` to control where data is processed.

In [ ]:
loader = AzureAIContentUnderstandingLoader(
    endpoint=endpoint,
    credential=credential,
    url=SAMPLE_PDF_URL,
    analyze_kwargs={"processing_location": "geography"},
)
docs = loader.load()
print(f"Loaded with geo-restricted processing: {docs[0].metadata['source']}")

---

## Summary

| Feature | Example |
|---------|--------|
| PDF / Document (URL) | `url="https://...invoice.pdf"` |
| Image / Chart | `url="https://...chart.jpg"` |
| Audio | `url="https://...call.mp3"` (segmentation **not** supported) |
| Video transcript | `url="https://...video.mp4"` (auto-selects `prebuilt-videoSearch`) |
| Video segmentation | `analyzer_id="my_video_segmenter"`, `output_mode="segment"` (requires custom `prebuilt-video` analyzer with `enableSegment`) |
| Custom doc analyzer | `analyzer_id="my_invoice_analyzer"` with `field_schema` (requires custom analyzer) |
| Document classification | `analyzer_id="my_doc_classifier"`, `output_mode="segment"` (requires custom `prebuilt-document` classifier) |
| Prebuilt field extraction | `analyzer_id="prebuilt-invoice"`, `model_deployments={...}` |
| Page-level splitting | `output_mode="page"` (documents only, no field extraction) |
| Partial analysis | `content_range="1-3"` |
| Local file | `file_path="invoice.pdf"` |
| Async loading | `await loader.aload()` |
| RAG pipeline | `output_mode="page"` -> split -> embed -> retrieve |
| Bytes input | `bytes_source=data, source="label"` |
| Advanced options | `analyze_kwargs={"processing_location": "geography"}` |

**Segmentation support:** `output_mode="segment"` works with **document** (`prebuilt-document`) and **video** (`prebuilt-video`) analyzers. Audio (`prebuilt-audio`, `prebuilt-callCenter`) does **not** support segmentation.

For more details, see the [Azure Content Understanding documentation](https://learn.microsoft.com/en-us/azure/ai-services/content-understanding/).